In [ ]:
# !pip install unidic-lite

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from torch import nn
from datasets import load_dataset
from transformers import AutoTokenizer

from collections import Counter
from tqdm.auto import tqdm, trange
import matplotlib.pyplot as plt


In [ ]:
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_default_device(device)

In [ ]:
ds = load_dataset("Verah/JParaCrawl-Filtered-English-Japanese-Parallel-Corpus");

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
ds["train"]

Dataset({
    features: ['id', 'english', 'japanese', 'model1_accepted', 'model2_accepted'],
    num_rows: 1000000
})

In [ ]:
jpn_tokenizer = AutoTokenizer.from_pretrained("cl-tohoku/bert-base-japanese")

In [ ]:
jpn_tokenizer.vocab_size

32000

In [ ]:
# jpn_smaple = cleaned_ds[0]["japanese"]
# jpn_tokenizer.encode(jpn_smaple)

In [ ]:
class Eng2JpnDataset(Dataset):
    def __init__(self, dataset, japanese_tokenizer, eager_tokenizer=False, max_length=512, mode="train"):
        if mode == "train":
            self.japanese_tokenizer = japanese_tokenizer
            self.eager_tokenizer = eager_tokenizer
            self.raw_data = dataset
            self.max_length = max_length
            self.eng_vocab = self._build_vocab(self.raw_data, "english")

            self.idx2word, self.word2idx = self._build_dict(self.eng_vocab)

            if self.eager_tokenizer:
                self.english_tokenized = []
                self.japanese_tokenized = []
                for english, japanese in zip(self.raw_data["english"], self.raw_data["japanese"]):
                    self.english_tokenized.append(self._eng_tokenize(english))
                    self.japanese_tokenized.append(self._jpn_tokenize(japanese))


    def _build_vocab(self, dataset, language, min_freq=10):
        vocab = Counter()
        for sentence in dataset[language]:
            words = [word.lower() for word in sentence.split()]
            vocab.update(words)

        vocab = [word for word, count in vocab.items() if count >= min_freq]
        return sorted(vocab)


    def _build_dict(self, vocab):
        word2idx = {word : idx+4 for idx, word in enumerate(vocab)}
        idx2word = {idx+4 : word for idx, word in enumerate(vocab)}

        word2idx["<PAD>"] = 0
        word2idx["<UNK>"] = 1
        word2idx["<BOS>"] = 2
        word2idx["<EOS>"] = 3

        idx2word[0] = "<PAD>"
        idx2word[1] = "<UNK>"
        idx2word[2] = "<BOS>"
        idx2word[3] = "<EOS>"

        return idx2word, word2idx


    def _eng_tokenize(self, eng_sentence):
        tokenized = []
        for word in eng_sentence.split():
            word = word.lower()
            tokenized.append(self.word2idx.get(word, self.word2idx["<UNK>"]))
        return tokenized

    def _jpn_tokenize(self, jpn_sentence):
        return self.japanese_tokenizer.encode(jpn_sentence, max_length=self.max_length, truncation=True)


    def __len__(self):
        return len(self.raw_data["english"])


    def __getitem__(self, index):
        if self.eager_tokenizer:
            return (self.english_tokenized[index], self.japanese_tokenized[index])
        else:
            eng, jpn = self._eng_tokenize(self.raw_data["english"][index]), self._jpn_tokenize(self.raw_data["japanese"][index])
            return (torch.tensor(eng[:self.max_length], dtype=torch.long), torch.tensor(jpn[:self.max_length], dtype=torch.long))


In [ ]:
cleaned_ds = ds["train"].filter(lambda x : x["model1_accepted"] == 1 or x["model2_accepted"] == 1)


In [ ]:
train_val_test = cleaned_ds.train_test_split(test_size=0.05, seed=42)
train_val = train_val_test["train"].train_test_split(test_size=0.1, seed=42)
test_dataset = train_val_test["test"]
val_dataset = train_val["test"]
train_dataset = train_val["train"]

In [ ]:
train_custom_dataset = Eng2JpnDataset(train_dataset, jpn_tokenizer)

In [ ]:
def collate_pad(batch):
    eng = pad_sequence(batch[0], batch_first=True)
    jpn = pad_sequence(batch[1], batch_first=True)
    return pack_padded_sequence(eng, torch.tensor(len(batch[0])), True, False),  pack_padded_sequence(eng,torch.tensor(len(batch[0])),True, False)
BATCH_SIZE = 32
train_loader = DataLoader(train_custom_dataset, BATCH_SIZE, True, collate_fn=collate_pad, pin_memory=True)
val_loader = DataLoader(val_dataset, BATCH_SIZE, True, collate_fn=collate_pad, pin_memory=True)
test_loader = DataLoader(test_dataset, BATCH_SIZE, True, collate_fn=collate_pad, pin_memory=True)

In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_dim, embed_dim=128, hidden_dim=256, BATCH_SIZE=32, num_layers=4):
        super().__init__()
        self.batch_size = BATCH_SIZE
        self.num_layers = num_layers
        self.embedding = nn.Embedding(vocab_dim, hidden_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers, batch_first=True, bidirectional=True)


    def forward(self, packed_input):
        embed = self.embedding(packed_input)
        output, (h_n, c_n) = self.lstm(embed)
        last_h = h_n.view(self.num_layers, 2, self.batch_size, -1)[-1]
        last_c = c_n.view(self.num_layers, 2, self.batch_size, -1)[-1]
        o_h = torch.cat([last_h[0], last_h[1]], dim=-1).squeeze(0) # B, 2*H
        o_c = torch.cat([last_c[0], last_c[1]], dim=-1).squeeze(0)
        return output (o_h, o_c)


class Decoder(nn.Module):
    def __init__(self, vocab_dim, embed_dim=128, hidden_dim=256, BATCH_SIZE=32, num_layers=4):
        super().__init__()
        self.hidden_dim = hidden_dim * 2
        self.embedding = nn.Embedding(vocab_dim, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_dim)


    def forward(self, encoder_o, bos=2, eos=3):
        embed = self.embedding(bos)
        h, c = encoder_o
        index_token_generated = []
        assert h.size(1) == self.hidden_dim or c.size(1) == self.hidden_dim, "encoder hidden state shape is not the same as decoder hidden state"
        while True:
            output, (h, c) = self.lstm(embed, (h, c))
            next_token = self.fc(h[-1]).argmax(dim=-1)
            index_token_generated.append(next_token)
            embed = self.embedding(next_token)
            if next_token == eos:
                break
        return index_token_generated


class Seq2Seq(nn.Module):
    def __init__(self, vocab_dim):
        super().__init__()
        self.encoder = Encoder(vocab_dim)
        self.decoder = Decoder(vocab_dim)
    def forward(self, text):
        _, (o_h, o_c) = self.encoder(text)
        translation = self.decoder((o_h, o_c))
        return translation

In [ ]:
def train_and_evaluate_one_epoch(model, optimizer, criterion, device, train_loader, val_loader): #assumes model on device
    model.train()
    for original_text, translation in tqdm(train_loader, leave=True):
        optimizer.zero_grad()
        pred_token = model(original_text)
        train_loss = criterion(pred_token, translation)
        train_loss.backward()
        optimizer.step()
    with torch.inference_mode()
        model.eval()
        for x_val, y_val in tqdm(val_loader, leave=False):
            pred_val = model(x_train)


    return avg_train_loss, avg_val_loss

